# Comparative Analysis of Machine Learning Models for Money Demand Forecasting — M1 (Narrow Money)

This notebook implements the M1 (narrow money demand) forecasting analysis from:

> Sikhwal S., Sen S. *Comparative Analysis of Machine Learning Models for Money Demand Forecasting in the Indian Economy.* HSE Economic Journal. 2024; 28(1): 133–158.

**Dataset:** Monthly time series, India, 1997–2021. Source: CEIC.  
**Target variable:** M1_real = M1SA / CPISA (real narrow money balances, log-differenced)  
**Features:** IPISA, Call money rate, NEER, BSE market capitalisation  
**Models:** AR(1) benchmark, Random Forest, Gradient Boosting, XGBoost, SVR, LASSO, LSTM  
**Validation:** Expanding window cross-validation with K = 2 to 7 folds  
**Evaluation:** MSE, RMSE, MAPE, SMAPE, TIC; Diebold-Mariano test with Harvey adjustment

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import statsmodels.api as sm
from statsmodels.tsa.ar_model import AutoReg
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from helper_functions import *

## 2. Data Loading and Preparation

### 2.1 Read the Excel file and select M1 variables

Variables selected as per the log-linearised Money Demand Function (Equation 1 in the paper):
- **M1SA** — seasonally adjusted nominal narrow money aggregate
- **CPISA** — seasonally adjusted CPI (price level, used to deflate M1SA to real balances)
- **IPISA** — Index of Industrial Production (income proxy, seasonally adjusted)
- **Call money rate** — interest rate variable for M1 (not log-transformed; in percentage form)
- **NEER** — Nominal Effective Exchange Rate
- **BSE mkt cap mn** — Bombay Stock Exchange market capitalisation (financial stability proxy)

In [2]:
df = pd.read_excel("DataMoneyDemand.xlsx")
df = df[['Date', 'M1SA', 'CPISA', 'IPISA', 'Call money rate', 'NEER', 'BSE mkt cap mn']]

df["Date"] = pd.to_datetime(df["Date"], format='%Y-%m-%d')
df = df.dropna()
df.set_index('Date', inplace=True)
df.head()

,M1SA,CPISA,IPISA,Call money rate,NEER,BSE mkt cap mn
Date,,,,,,
1997-01-01,2.292705e+06,43.322394,51.012669,4.84,125.76,4582610.0
1997-02-01,2.319171e+06,43.616447,51.179397,5.08,128.69,4846240.0
1997-03-01,2.336500e+06,43.906066,47.080132,4.35,129.66,4639150.0
1997-04-01,2.348801e+06,44.082589,56.884312,1.22,130.39,5020820.0
1997-05-01,2.382663e+06,43.838752,50.131279,5.90,129.56,5063910.0


### 2.2 Compute real money balances and apply log transformation

Following Equation 1 in the paper, real money balances are computed by dividing the seasonally
adjusted nominal money stock (M1SA) by the seasonally adjusted price level (CPISA).
Natural log is then applied to all variables **except** the interest rate (Call money rate),
which is already in percentage form.

In [3]:
# Compute real money balances: M1_real = M1SA / CPISA (Equation 1 in the paper)
df['M1_real'] = df['M1SA'] / df['CPISA']

# Drop the nominal M1SA and CPISA columns — no longer needed
df = df.drop(columns=['M1SA', 'CPISA'])

# Apply log to all variables except Call money rate (interest rate, in percentage form)
df[["M1_real", "IPISA", "NEER", "BSE mkt cap mn"]] =     df[["M1_real", "IPISA", "NEER", "BSE mkt cap mn"]].apply(np.log)
df.head()

,IPISA,Call money rate,NEER,BSE mkt cap mn,M1_real
Date,,,,,
1997-01-01,3.932074,4.84,4.834375,15.337779,10.876573
1997-02-01,3.935337,5.08,4.857406,15.393714,10.881286
1997-03-01,3.851851,4.35,4.864916,15.350042,10.882112
1997-04-01,4.041020,1.22,4.870530,15.429104,10.883350
1997-05-01,3.914645,5.90,4.864144,15.437649,10.903211


## 3. Stationarity Tests

### 3.1 Unit root tests at level (ADF and Phillips-Perron)

The null hypothesis for both ADF and PP tests is that the series has a unit root (non-stationary).
Maximum lag length: 15

In [4]:
print("=== Unit Root Tests: Level ===")
adf_test(df)

=== Unit Root Tests: Level ===
IPISA:
  ADF Statistic : -1.8106
  p-value       : 0.3752
  Critical Value (1%): -3.4527
  Critical Value (5%): -2.8714
  Critical Value (10%): -2.5720
  Phillips-Perron:      Phillips-Perron Test (Z-tau)    
Test Statistic                 -2.067
P-value                         0.258
Lags                               16
-------------------------------------

Trend: Constant
Critical Values: -3.45 (1%), -2.87 (5%), -2.57 (10%)
Null Hypothesis: The process contains a unit root.
Alternative Hypothesis: The process is weakly stationary.

Call money rate:
  ADF Statistic : -5.5587
  p-value       : 0.0000
  Critical Value (1%): -3.4525
  Critical Value (5%): -2.8713
  Critical Value (10%): -2.5720
  Phillips-Perron:      Phillips-Perron Test (Z-tau)    
Test Statistic                 -9.675
P-value                         0.000
Lags                               16
-------------------------------------

Trend: Constant
Critical Values: -3.45 (1%), -2.87 (5%),

### 3.2 First differencing and re-testing

All series are non-stationary at level. We apply first differencing to obtain I(1) stationarity.

In [5]:
differenced_data = df.diff(axis=0).dropna()
assert differenced_data.shape == (len(df) - 1, df.shape[1])

print("=== Unit Root Tests: First Difference ===")
adf_test(differenced_data)

=== Unit Root Tests: First Difference ===
IPISA:
  ADF Statistic : -14.1495
  p-value       : 0.0000
  Critical Value (1%): -3.4527
  Critical Value (5%): -2.8714
  Critical Value (10%): -2.5720
  Phillips-Perron:      Phillips-Perron Test (Z-tau)    
Test Statistic                -35.215
P-value                         0.000
Lags                               16
-------------------------------------

Trend: Constant
Critical Values: -3.45 (1%), -2.87 (5%), -2.57 (10%)
Null Hypothesis: The process contains a unit root.
Alternative Hypothesis: The process is weakly stationary.

Call money rate:
  ADF Statistic : -9.9291
  p-value       : 0.0000
  Critical Value (1%): -3.4529
  Critical Value (5%): -2.8715
  Critical Value (10%): -2.5721
  Phillips-Perron:      Phillips-Perron Test (Z-tau)    
Test Statistic                -39.363
P-value                         0.000
Lags                               16
-------------------------------------

Trend: Constant
Critical Values: -3.45 (1%),

## 4. Data Preparation for Modelling

### 4.1 Construct the M1 modelling dataframe and apply log-differencing

In [6]:
# Rename M1_real to y1 for modelling
M1_df = df.assign(y1=df["M1_real"]).drop("M1_real", axis=1)

# Store the 2018-01-01 level value of M1_real for inverse transformation later
temp = df.loc[pd.to_datetime('2018-01-01')]["M1_real"]

# Apply first difference (log-difference = continuous growth rate)
M1_df = M1_df.diff().dropna()
M1_df.head()

,IPISA,Call money rate,NEER,BSE mkt cap mn,y1
Date,,,,,
1997-02-01,0.003263,0.24,0.023031,0.055934,0.004713
1997-03-01,-0.083486,-0.73,0.007509,-0.043672,0.000826
1997-04-01,0.189169,-3.13,0.005614,0.079062,0.001238
1997-05-01,-0.126374,4.68,-0.006386,0.008546,0.019861
1997-06-01,0.014460,-0.74,0.000386,0.150261,0.020093


### 4.2 Train/test split

The dataset is split at January 2018: ~80% training (1997–2017), ~20% testing (2018–2021), yielding N=48 test observations.

In [7]:
X = M1_df[["BSE mkt cap mn", "Call money rate", "IPISA", "NEER"]].to_numpy()
y = np.array(M1_df['y1'])

X, X_test, y, y_test, idx = make_split(M1_df, X, y, '2018-01-01')
print(f"X_train shape : {X.shape}")
print(f"X_test shape  : {X_test.shape}")
print(f"y_train shape : {y.shape}")
print(f"y_test shape  : {y_test.shape}")

X_train shape : (251, 4)
X_test shape  : (48, 4)
y_train shape : (251,)
y_test shape  : (48,)


## 5. Model Training and Evaluation

### 5.1 Helper: evaluate_model_for_split

Trains a given sklearn-compatible model using expanding window cross-validation across K = 2 to 7 folds, predicts on the test set, reverses the log-difference transformation, and prints evaluation metrics.

In [8]:
def evaluate_model_for_split(splits, train_X, train_Y, test_X, test_Y, model_name, *args, **kwargs):
    """Train model across expanding window folds and evaluate on test set.

    For each split K, the model is trained on an incrementally expanding
    training window, then evaluated on the held-out test set. Predictions
    are reverse-transformed from log-differences back to levels using the
    2018-01-01 anchor value (temp).

    Returns:
        y_preds: list of untransformed predictions, one per split
    """
    y_preds = []
    print(model_name)
    for split in splits:
        model = model_name(*args, **kwargs)
        _, model = expanding_window_cv_with_splits(train_X, train_Y, model, num_splits=split)
        y_pred_diff = model.predict(test_X)

        # Reverse log-difference transformation
        y_pred_untransformed = np.cumsum(y_pred_diff[1:]) + temp
        y_test_untransformed = np.cumsum(test_Y[1:]) + temp

        mean_rmse, mean_mse, mean_mape, mean_smape, mean_tic = metrics(
            y_test_untransformed, y_pred_untransformed)
        print_metrics(mean_rmse, mean_mse, mean_mape, mean_smape, mean_tic, split)
        y_preds.append(y_pred_untransformed)

    return y_preds

### 5.2 AR(1) — Benchmark Model

The AR(1) model serves as the benchmark. It is fitted on each expanding window fold and predicts directly on the test set.

In [9]:
splits = [2, 3, 4, 5, 6, 7]

for split in splits:
    result = AR_expanding_window_cv_with_splits(X, y, n_splits=split)
    mean_rmse, mean_mse, mean_mape, mean_smape, mean_tic = [], [], [], [], []
    print(f"K = {split}")

    for (X_train, y_train, X_val, y_val) in result:
        AR = AutoReg(y_train, lags=1).fit()
        y_pred_ar = AR.predict(start=len(y_val), end=len(y_val) + len(y_test) - 1)

        mean_rmse.append(np.sqrt(mean_squared_error(y_test, y_pred_ar)))
        mean_mse.append(mean_squared_error(y_test, y_pred_ar))
        mean_mape.append(mean_absolute_percentage_error(y_test, y_pred_ar))
        mean_smape.append(symmetric_mean_absolute_percentage_error(y_test, y_pred_ar))
        mean_tic.append(theil_inequality_coeff(y_test, y_pred_ar))

        y_pred_untransformed_ar = np.cumsum(y_pred_ar[1:]) + temp
        y_test_untransformed = np.cumsum(y_test[1:]) + temp

    mean_rmse, mean_mse, mean_mape, mean_smape, mean_tic = metrics(
        y_test_untransformed, y_pred_untransformed_ar)
    print_metrics(mean_rmse, mean_mse, mean_mape, mean_smape, mean_tic, split)

K = 2
Test Set for K = 2
Mean of MSE   : 0.00491
Mean of RMSE  : 0.07009
Mean of MAPE  : 0.49311
Mean of SMAPE : 0.49471
Mean of TIC   : 0.00286
 
K = 3
Test Set for K = 3
Mean of MSE   : 0.00497
Mean of RMSE  : 0.07047
Mean of MAPE  : 0.49579
Mean of SMAPE : 0.49742
Mean of TIC   : 0.00287
 
K = 4
Test Set for K = 4
Mean of MSE   : 0.00484
Mean of RMSE  : 0.06958
Mean of MAPE  : 0.48958
Mean of SMAPE : 0.49116
Mean of TIC   : 0.00284
 
K = 5
Test Set for K = 5
Mean of MSE   : 0.00491
Mean of RMSE  : 0.07009
Mean of MAPE  : 0.49311
Mean of SMAPE : 0.49471
Mean of TIC   : 0.00286
 
K = 6
Test Set for K = 6
Mean of MSE   : 0.00553
Mean of RMSE  : 0.07436
Mean of MAPE  : 0.52347
Mean of SMAPE : 0.52528
Mean of TIC   : 0.00303
 
K = 7
Test Set for K = 7
Mean of MSE   : 0.00545
Mean of RMSE  : 0.07385
Mean of MAPE  : 0.51987
Mean of SMAPE : 0.52166
Mean of TIC   : 0.00301
 


### 5.3 Random Forest Regression

In [13]:
splits = [2, 3, 4, 5, 6, 7]

y_pred_random_forest = evaluate_model_for_split(
    splits, X, y, X_test, y_test, RandomForestRegressor)

<class 'sklearn.ensemble._forest.RandomForestRegressor'>
Test Set for K = 2
Mean of MSE   : 0.00472
Mean of RMSE  : 0.06869
Mean of MAPE  : 0.50406
Mean of SMAPE : 0.50561
Mean of TIC   : 0.00280
 
Test Set for K = 3
Mean of MSE   : 0.00263
Mean of RMSE  : 0.05133
Mean of MAPE  : 0.37657
Mean of SMAPE : 0.37743
Mean of TIC   : 0.00209
 
Test Set for K = 4
Mean of MSE   : 0.00654
Mean of RMSE  : 0.08087
Mean of MAPE  : 0.60757
Mean of SMAPE : 0.60972
Mean of TIC   : 0.00330
 
Test Set for K = 5
Mean of MSE   : 0.00250
Mean of RMSE  : 0.05001
Mean of MAPE  : 0.36254
Mean of SMAPE : 0.36337
Mean of TIC   : 0.00204
 
Test Set for K = 6
Mean of MSE   : 0.00294
Mean of RMSE  : 0.05426
Mean of MAPE  : 0.39272
Mean of SMAPE : 0.39368
Mean of TIC   : 0.00221
 
Test Set for K = 7
Mean of MSE   : 0.00525
Mean of RMSE  : 0.07242
Mean of MAPE  : 0.54080
Mean of SMAPE : 0.54253
Mean of TIC   : 0.00295
 


### 5.4 Gradient Boosting

In [14]:
splits = [2, 3, 4, 5, 6, 7]

y_pred_gradboost = evaluate_model_for_split(
    splits, X, y, X_test, y_test, GradientBoostingRegressor)

<class 'sklearn.ensemble._gb.GradientBoostingRegressor'>
Test Set for K = 2
Mean of MSE   : 0.00787
Mean of RMSE  : 0.08871
Mean of MAPE  : 0.65112
Mean of SMAPE : 0.65371
Mean of TIC   : 0.00362
 
Test Set for K = 3
Mean of MSE   : 0.00706
Mean of RMSE  : 0.08404
Mean of MAPE  : 0.61959
Mean of SMAPE : 0.62192
Mean of TIC   : 0.00343
 
Test Set for K = 4
Mean of MSE   : 0.00899
Mean of RMSE  : 0.09481
Mean of MAPE  : 0.70210
Mean of SMAPE : 0.70506
Mean of TIC   : 0.00387
 
Test Set for K = 5
Mean of MSE   : 0.00752
Mean of RMSE  : 0.08672
Mean of MAPE  : 0.63554
Mean of SMAPE : 0.63801
Mean of TIC   : 0.00354
 
Test Set for K = 6
Mean of MSE   : 0.00989
Mean of RMSE  : 0.09944
Mean of MAPE  : 0.74087
Mean of SMAPE : 0.74413
Mean of TIC   : 0.00406
 
Test Set for K = 7
Mean of MSE   : 0.00731
Mean of RMSE  : 0.08551
Mean of MAPE  : 0.63208
Mean of SMAPE : 0.63449
Mean of TIC   : 0.00349
 


### 5.5 XGBoost

In [15]:
splits = [2, 3, 4, 5, 6, 7]

y_pred_xgboost = evaluate_model_for_split(
    splits, X, y, X_test, y_test, XGBRegressor)

<class 'xgboost.sklearn.XGBRegressor'>
Test Set for K = 2
Mean of MSE   : 0.00447
Mean of RMSE  : 0.06689
Mean of MAPE  : 0.46678
Mean of SMAPE : 0.46824
Mean of TIC   : 0.00273
 
Test Set for K = 3
Mean of MSE   : 0.00520
Mean of RMSE  : 0.07209
Mean of MAPE  : 0.51946
Mean of SMAPE : 0.52117
Mean of TIC   : 0.00294
 
Test Set for K = 4
Mean of MSE   : 0.00936
Mean of RMSE  : 0.09673
Mean of MAPE  : 0.67869
Mean of SMAPE : 0.68176
Mean of TIC   : 0.00395
 
Test Set for K = 5
Mean of MSE   : 0.00447
Mean of RMSE  : 0.06689
Mean of MAPE  : 0.46678
Mean of SMAPE : 0.46824
Mean of TIC   : 0.00273
 
Test Set for K = 6
Mean of MSE   : 0.00769
Mean of RMSE  : 0.08770
Mean of MAPE  : 0.61191
Mean of SMAPE : 0.61443
Mean of TIC   : 0.00358
 
Test Set for K = 7
Mean of MSE   : 0.00996
Mean of RMSE  : 0.09982
Mean of MAPE  : 0.68732
Mean of SMAPE : 0.69059
Mean of TIC   : 0.00407
 


### 5.6 Support Vector Regression (SVR)

In [16]:
splits = [2, 3, 4, 5, 6, 7]

y_pred_svr = evaluate_model_for_split(
    splits, X, y, X_test, y_test, SVR)

<class 'sklearn.svm._classes.SVR'>
Test Set for K = 2
Mean of MSE   : 0.40700
Mean of RMSE  : 0.63797
Mean of MAPE  : 4.49069
Mean of SMAPE : 4.62798
Mean of TIC   : 0.02653
 
Test Set for K = 3
Mean of MSE   : 0.40664
Mean of RMSE  : 0.63768
Mean of MAPE  : 4.48874
Mean of SMAPE : 4.62590
Mean of TIC   : 0.02652
 
Test Set for K = 4
Mean of MSE   : 0.40628
Mean of RMSE  : 0.63740
Mean of MAPE  : 4.48678
Mean of SMAPE : 4.62382
Mean of TIC   : 0.02651
 
Test Set for K = 5
Mean of MSE   : 0.40700
Mean of RMSE  : 0.63797
Mean of MAPE  : 4.49069
Mean of SMAPE : 4.62798
Mean of TIC   : 0.02653
 
Test Set for K = 6
Mean of MSE   : 0.40556
Mean of RMSE  : 0.63684
Mean of MAPE  : 4.48288
Mean of SMAPE : 4.61967
Mean of TIC   : 0.02648
 
Test Set for K = 7
Mean of MSE   : 0.40520
Mean of RMSE  : 0.63656
Mean of MAPE  : 4.48092
Mean of SMAPE : 4.61759
Mean of TIC   : 0.02647
 


### 5.7 LASSO Regression

In [17]:
splits = [2, 3, 4, 5, 6, 7]

y_pred_lasso = evaluate_model_for_split(
    splits, X, y, X_test, y_test, Lasso,
    fit_intercept=False, max_iter=20000, tol=1e-2, precompute=True, alpha=1)

<class 'sklearn.linear_model._coordinate_descent.Lasso'>
Test Set for K = 2
Mean of MSE   : 0.04045
Mean of RMSE  : 0.20113
Mean of MAPE  : 1.41856
Mean of SMAPE : 1.43188
Mean of TIC   : 0.00824
 
Test Set for K = 3
Mean of MSE   : 0.04045
Mean of RMSE  : 0.20113
Mean of MAPE  : 1.41856
Mean of SMAPE : 1.43188
Mean of TIC   : 0.00824
 
Test Set for K = 4
Mean of MSE   : 0.04045
Mean of RMSE  : 0.20113
Mean of MAPE  : 1.41856
Mean of SMAPE : 1.43188
Mean of TIC   : 0.00824
 
Test Set for K = 5
Mean of MSE   : 0.04045
Mean of RMSE  : 0.20113
Mean of MAPE  : 1.41856
Mean of SMAPE : 1.43188
Mean of TIC   : 0.00824
 
Test Set for K = 6
Mean of MSE   : 0.04045
Mean of RMSE  : 0.20113
Mean of MAPE  : 1.41856
Mean of SMAPE : 1.43188
Mean of TIC   : 0.00824
 
Test Set for K = 7
Mean of MSE   : 0.04045
Mean of RMSE  : 0.20113
Mean of MAPE  : 1.41856
Mean of SMAPE : 1.43188
Mean of TIC   : 0.00824
 


### 5.8 LSTM (Long Short-Term Memory)

#### 5.8.1 Device setup, model definition, and test tensor preparation

In [18]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Prepare test tensors for LSTM evaluation
X_test_tensor = torch.from_numpy(X_test).float().to(device).unsqueeze(0)
y_test_tensor = torch.from_numpy(y_test).float().to(device).unsqueeze(0)


class LSTMModel(nn.Module):
    """LSTM model for time series forecasting."""
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.dropout = dropout
        if self.num_layers == 1:
            self.lstm = nn.LSTM(input_size, hidden_size, 1, batch_first=True)
        else:
            self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                                batch_first=True, dropout=self.dropout)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out)
        return out.squeeze(-1)


def accuracy_measures(y_test, predictions_test):
    """Print evaluation metrics for LSTM predictions (accepts tensors)."""
    y_test = y_test.cpu().numpy()
    predictions_test = predictions_test.cpu().numpy()
    mse = mean_squared_error(y_test, predictions_test)
    print("RMSE :", np.sqrt(mse))
    print("MSE  :", mse)
    print("MAPE :", mean_absolute_percentage_error(y_test, predictions_test))
    print("SMAPE:", symmetric_mean_absolute_percentage_error(y_test, predictions_test))
    print("TIC  :", theil_inequality_coeff(y_test, predictions_test))
    print("")

#### 5.8.2 LSTM expanding window cross-validation function

In [19]:
def tf_expanding_window_cv_with_splits(X, y, num_epochs, criterion, n_splits=5,
                                       input_size=4, hidden_size=64, num_layers=3,
                                       dropout=0.2, learning_rate=0.1, device='cpu'):
    """Train LSTM with expanding window cross-validation using TimeSeriesSplit.

    For each fold, a fresh LSTM is instantiated and trained on the expanding
    training window. The best model (lowest validation loss) is returned.
    Trained for num_epochs epochs per fold with SGD optimiser.
    """
    tsv = TimeSeriesSplit(n_splits=n_splits)
    batch_size = 32

    for train_idx, val_idx in tsv.split(X):
        X_train_fold = torch.from_numpy(X[train_idx]).float().to(device).unsqueeze(0)
        y_train_fold = torch.from_numpy(y[train_idx]).float().to(device).unsqueeze(0)
        X_val_fold   = torch.from_numpy(X[val_idx]).float().to(device).unsqueeze(0)
        y_val_fold   = torch.from_numpy(y[val_idx]).float().to(device).unsqueeze(0)

        lstm_model = LSTMModel(input_size, hidden_size, num_layers, 1, dropout).to(device)
        optimizer = torch.optim.SGD(lstm_model.parameters(), lr=learning_rate)

        train_loader = DataLoader(
            TensorDataset(X_train_fold, y_train_fold),
            batch_size=batch_size, shuffle=True)

        lstm_model.train()
        for epoch in range(num_epochs):
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = lstm_model(inputs)
                loss = criterion(outputs, labels)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        lstm_model.eval()
        with torch.no_grad():
            predictions = lstm_model(X_val_fold)
        val_loss = criterion(y_val_fold, predictions)

    return val_loss, lstm_model

#### 5.8.3 Hyperparameter grid search and test set evaluation

Exhaustive grid search over hidden size, number of layers, dropout rate, and learning rate for each fold K = 2 to 7. Best hyperparameters selected by lowest validation MSE. LSTM trained for **300 epochs**.

In [20]:
num_splits = [2, 3, 4, 5, 6, 7]
criterion  = nn.MSELoss()
best_hyperparameters = []

for split in num_splits:
    best_val_loss_for_split = np.inf
    best_hyperparams_for_split = {}

    for hidden_size in [32, 64, 128]:
        for num_layers in [1, 2, 3]:
            for dropout_rate in [0.0, 0.1, 0.2]:
                for learning_rate in [0.0001, 0.0005, 0.0008, 0.003]:
                    val_loss, model = tf_expanding_window_cv_with_splits(
                        X, y, 300, criterion,
                        n_splits=split, input_size=4,
                        hidden_size=hidden_size, num_layers=num_layers,
                        dropout=dropout_rate, learning_rate=learning_rate,
                        device=device)

                    if val_loss < best_val_loss_for_split:
                        best_val_loss_for_split = val_loss
                        best_hyperparams_for_split = {
                            'hidden_size': hidden_size,
                            'num_layers': num_layers,
                            'dropout_rate': dropout_rate,
                            'learning_rate': learning_rate,
                            'split_size': split}

    best_hyperparameters.append(best_hyperparams_for_split)

    print(f"Split K = {split} | Best params: {best_hyperparams_for_split}")

    # Evaluate best model on test set
    model.eval()
    with torch.no_grad():
        predictions_test = model(X_test_tensor)

    y_pred_untransformed = np.cumsum(predictions_test.flatten().cpu().numpy()[1:]) + temp
    y_test_untransformed_lstm = np.cumsum(y_test_tensor.flatten().cpu().numpy()[1:]) + temp

    y_pred_untransformed_t = torch.tensor(y_pred_untransformed)
    y_test_untransformed_t = torch.tensor(y_test_untransformed_lstm)

    print(f"Test loss (MSE on levels): {criterion(y_test_untransformed_t, y_pred_untransformed_t).item():.5f}")
    accuracy_measures(y_test_untransformed_t, y_pred_untransformed_t)

print("\nBest hyperparameters per fold:")
for h in best_hyperparameters:
    print(h)

# Capture K=6 predictions for DM test
    if split == 6:
        y_pred_lstm = y_pred_untransformed
        y_test_untransformed = y_test_untransformed_lstm

KeyboardInterrupt: 

## 6. Diebold-Mariano Test — Pairwise Forecast Comparison

The DM test (with Harvey adjustment) compares pairwise forecast accuracy across all models for K = 6 (as reported in Table 4 of the paper). A positive DM statistic indicates the row model forecasts better than the column model; a negative value indicates the opposite. Significance: \* 10%, \*\* 5%, \*\*\* 1%.

In [21]:
# Collect all K=6 predictions (index 4 in the splits list [2,3,4,5,6,7])
k6_idx = 4  # K=6 is the 5th element (0-indexed)

model_names = ['AR(1)', 'RF', 'GB', 'XGBoost', 'SVR', 'LASSO', 'LSTM']
preds = {
    'AR(1)' : y_pred_untransformed_ar,
    'RF'    : y_pred_random_forest[k6_idx],
    'GB'    : y_pred_gradboost[k6_idx],
    'XGBoost': y_pred_xgboost[k6_idx],
    'SVR'   : y_pred_svr[k6_idx],
    'LASSO' : y_pred_lasso[k6_idx],
    'LSTM'  : y_pred_lstm,
}

def significance_stars(p):
    if p < 0.01:  return '***'
    elif p < 0.05: return '**'
    elif p < 0.10: return '*'
    else:          return ''

print("Diebold-Mariano Test Matrix (K=6, Harvey adjustment, h=1)")
print("Rows = pred1, Cols = pred2")
print()

header = f"{'':<10}" + "".join(f"{n:>14}" for n in model_names)
print(header)
print("-" * len(header))

for row_name in model_names:
    row_str = f"{row_name:<10}"
    for col_name in model_names:
        if row_name == col_name:
            row_str += f"{'–':>14}"
        else:
            result = dm_test(y_test_untransformed,
                             preds[row_name], preds[col_name],
                             h=1, harvey_adj=True)
            stars = significance_stars(result.p_value)
            cell = f"{result.DM:.3f}{stars}"
            row_str += f"{cell:>14}"
    print(row_str)

print()
print("Note: * p<0.10, ** p<0.05, *** p<0.01")

NameError: name 'y_pred_lstm' is not defined